## Generate Eval dataset

In [0]:
%run ../config

In [0]:
spark.sql(f'use catalog {catalog}')
spark.sql(f'use schema {dbName}')

Generate a synthetic evaluation dataset for customer data and billing questions.

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, pandas_udf, concat_ws, lit, col, from_json, struct, expr, regexp_replace
from pyspark.sql.types import StringType, ArrayType
import pandas as pd

# 1단계: 고객 20명 데이터 로드
df = spark.table("customers").limit(20).withColumn("row_id", monotonically_increasing_id())

# 2단계: 20개 템플릿을 사용하여 질문 생성
@pandas_udf(StringType())
def generate_question(email: pd.Series, row_id: pd.Series) -> pd.Series:
    templates = [
        "{email} 사용자의 연락처는 무엇인가요?",
        "{email}의 전체 주문 내역을 보여주세요.",
        "{email}의 현재 구독 상태를 알려주세요.",
        "{email}의 결제 상세 정보를 조회해 주세요.",
        "{email}의 미납된 인보이스가 있나요?",
        "{email}의 구매 제품은 무엇인가요",
        "{email}의 멤버십 등급이 어떻게 되나요?",
        "{email}의 가입일은 언제인가요?",
        "{email}의 구독 현황 요약본을 보여주세요.",
        "{email}의 거주 도시(시/군/구)는 어디인가요?",
        "{email}의 현재 계정 상태를 알려주세요.",
        "{email}의 고객 유지 기간은 얼마나 되었나요?",
        "{email}의 이탈 위험 점수는 몇 점인가요?",
        "{email}의 고객 가치 점수를 알려주세요.",
        "{email} 계정에 자동 결제가 설정되어 있나요?",
        "{email}의 연체 횟수는 총 몇 건인가요?",
        "{email}의 우편번호는 무엇인가요?",
        "{email}의 고객 유형은 무엇인가요? (개인/비즈니스 등)",
        "{email}의 전체 주소를 알려주세요."
    ]
    return pd.Series([
        templates[int(i) % len(templates)].format(email=e)
        for i, e in zip(row_id, email)
    ])

df = df.withColumn("question", generate_question("email", "row_id"))

# 3단계: Pandas로 변환 후 다시 Spark로 변환하여 UDF 구체화 완료
df_pd = df.toPandas()
df_clean = spark.createDataFrame(df_pd)

# 4단계: AI_QUERY를 위한 프롬프트 구성 (순수 JSON 반환 요청)
df_clean = df_clean.withColumn(
    "prompt",
    concat_ws(
        " ",
        lit("당신은 현재 AI 시스템을 평가하고 있습니다."),
        lit("다음 고객 기록을 참고하세요:"),
        concat_ws(", ",
            df_clean.first_name, df_clean.last_name, df_clean.email, df_clean.phone,
            df_clean.address, df_clean.city, df_clean.state, df_clean.zip_code,
            df_clean.customer_segment, df_clean.registration_date.cast("string"),
            df_clean.customer_status, df_clean.loyalty_tier,
            df_clean.tenure_years.cast("string"), df_clean.churn_risk_score.cast("string"),
            df_clean.customer_value_score.cast("string")
        ),
        lit("질문의 답변에 포함되어야 하는 핵심 사실(expected_facts)을 JSON 배열로 작성하세요."),
        lit("각 요소는 자연스러운 완성형 문장이어야 합니다."),
        lit("중요: 마크다운 코드 블록(```json)을 사용하지 말고, 순수한 JSON 배열만 반환하세요."),
        lit("예시: [\"사실 1\", \"사실 2\"]"),
        lit("Question:"), df_clean.question
    )
)

# 5단계: 뷰 등록 및 AI_QUERY 호출
df_clean.createOrReplaceTempView("customer_test_questions")

final_df_raw = spark.sql("""
SELECT 
  question,
  AI_QUERY("databricks-gemma-3-12b", prompt) AS expected_facts_json
FROM customer_test_questions
""")

# 6단계: 마크다운 코드 블록 제거 및 JSON 문자열을 Array<String>으로 파싱
final_df = final_df_raw.withColumn(
    "expected_facts_json_clean",
    regexp_replace(
        regexp_replace(col("expected_facts_json"), "```json\\s*", ""),
        "\\s*```", ""
    )
).withColumn(
    "expected_facts",
    from_json(col("expected_facts_json_clean"), ArrayType(StringType()))
)

# 7단계: 평가용 구조화된 데이터 형식 구성
eval_df = final_df.withColumn("inputs", struct("question")) \
                  .withColumn("predictions", lit("")) \
                  .withColumn("expectations", struct("expected_facts")) \
                  .select("inputs", "predictions", "expectations")

# 8단계: 저장
eval_df.write.format('json').mode("overwrite").save(f"/Volumes/{catalog}/{dbName}/{volume_name}/eval_dataset")

In [0]:
display(eval_df)